# 2 — Guide du `HighFrequencyImputer` : l'influence de ses paramètres

`HighFrequencyImputer` impute les colonnes observées à basse fréquence sur une grille plus fine,
en cascade, avec un suivi complet de la provenance de chaque cellule produite.

Ce notebook est un **guide de prise en main**.

## Les deux axes

L'espace des paramètres repose essentiellement sur **deux axes orthogonaux**, chacun répondant à une question :

| Axe | Paramètre | Question |
|-----|-----------|----------|
| **Axe 1** | `covariate_strategy` | Comment une **covariable** observée moins souvent que la grille courante est-elle rendue disponible au modèle ? |
| **Axe 2** | `impute_intermediate_frequencies` | La variable imputée traverse-t-elle des **fréquences intermédiaires**, et son modèle final s'entraîne-t-il sur ses propres imputations ? |

L'axe 1 gouverne les **colonnes** tendues à l'estimateur, l'axe 2 gouverne les **lignes** de sa
cible. Ils se composent sans se connaître. Ce notebook les traite séparément (sections 4 et 5),
puis les croise (section 6).

## Table des matières

1. [Importation des modules](#1)
2. [Les jeux de données](#2)
3. [Première prise en main](#3)
4. [**Axe 1** — la matérialisation des covariables](#4)
5. [**Axe 2** — les fréquences intermédiaires](#5)
6. [Le croisement des deux axes](#6)
7. [Les autres paramètres](#7)
8. [Intégration dans un workflow de prédiction](#8)
9. [Récapitulatif](#9)

<a id="1"></a>
## 1 — Importation des modules

In [ ]:
# Importation des modules
# Modules de base
import time
import warnings

# Manipulation de données
import numpy as np
import pandas as pd

# Graphiques
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap

# Sklearn
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# Module tsforecast
from tsforecast.crossvals import TSOutOfSampleSplit
from tsforecast.delays import PublicationDelayTransformer
from tsforecast.frequency import HighFrequencyImputer
from tsforecast.frequency.covariate_materializer import CovariateMaterializer
from tsforecast.frequency.provenance import ProvenanceType
from tsforecast.utils.frequency import detect_dataset_frequency
from tsforecast.xy import XYPipeline

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Chronomètre du notebook
NOTEBOOK_START = time.perf_counter()

# Affichage
print("Modules importés avec succès !")

<a id="2"></a>
## 2 — Les jeux de données

**Le jeu de séries temporelles** représente les indicateurs macroéconomiques d'un pays :

| Variable | Fréquence | Particularité |
|---|---|---|
| `production_industrielle` | mensuelle | historique tronqué (commence en 2019) |
| `inflation_ipc` | mensuelle | dernière valeur retirée (délai de publication) |
| `taux_chomage` | mensuelle | dernière valeur retirée |
| `pib_trimestriel` | **trimestrielle** | publiée aux mois 1, 4, 7, 10 |
| `balance_commerciale_annuelle` | **annuelle** | historique remontant à 2015, **avant** le début des mensuelles : l'index global en devient irrégulier |

**Le jeu de panel** porte les mêmes indicateurs pour trois pays, avec trois difficultés
supplémentaires pour l'imputeur :

1. **Couverture temporelle hétérogène** — chaque pays a son propre début et sa propre fin.
2. **Fréquence hétérogène pour une même variable** — `depenses_publiques_pib` est annuelle pour la
   France et l'Italie, **trimestrielle** pour l'Allemagne. Les fréquences détectées sont indexées
   par couple `(entité, colonne)`, précisément pour ce cas.
3. **Variable structurellement absente pour une entité** — `climat_affaires` n'est jamais observée
   pour l'Italie. Cette situation est adressée par le paramètre `covariate_eligibility` (section 4.4).

### 2.1 — Le jeu de séries temporelles

In [ ]:
# Fonction de création de séries temporelles
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Création de l'index mensuel
    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    # Initialisation du DataFrame
    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # ----- Variables mensuelles -----
    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC, entre 0.5% et 4%)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel, entre 5% et 12%)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),  # Choc économique
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # ----- Variable trimestrielle : PIB -----
    # Le PIB n'est disponible qu'aux fins de trimestre
    pib_base = 2500
    pib_growth_quarterly = 0.5  # Croissance trimestrielle moyenne
    df['pib_trimestriel'] = np.nan

    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # ----- Variable annuelle : Balance commerciale -----
    # Historique disponible dès `annual_start_date`, antérieur au début des
    # variables mensuelles : l'index temporel global en devient irrégulier
    # (quelques observations annuelles isolées avant le début de la grille mensuelle).
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # ----- Simulation des délais de publication -----
    # Délai de 1 mois pour l'inflation et le chômage
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    # Délai de 2 mois pour le PIB
    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    # Délai de 3 mois pour la balance commerciale annuelle
    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # ----- Simulation de données historiques limitées -----
    # La production industrielle n'est disponible qu'à partir de 2019
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df

### 2.2 — Le jeu de panel

In [ ]:
# Fonction de création d'un jeu de données de panel fictif
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    # Initialisation du seed
    np.random.seed(seed)

    # Définition des pays et leurs caractéristiques
    # Chaque pays possède sa propre période de couverture (start_date / end_date)
    # ainsi que sa propre fréquence de publication pour les dépenses publiques
    countries = {
        'France': {
            'climat_affaires_observe': True,  # Italie : jamais observée
            'pib_base': 2800,
            'inflation_base': 1.5,
            'chomage_base': 8.0,
            'depenses_base': 55.0,
            'start_date': '2018-01-01',
            'end_date': '2024-07-01',
            'prod_ind_start': '2018-06-01',  # Historique complet
            'depenses_frequency': 'annuelle',
            'annual_start_date': '2015-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        },
        'Allemagne': {
            'climat_affaires_observe': True,  # Italie : jamais observée
            'pib_base': 3500,
            'inflation_base': 1.2,
            'chomage_base': 5.5,
            'depenses_base': 45.0,
            'start_date': '2018-07-01',  # Début plus tardif que la France
            'end_date': '2024-04-01',  # Fin plus précoce que la France
            'prod_ind_start': '2019-01-01',  # Historique partiel
            'depenses_frequency': 'trimestrielle',
            'annual_start_date': '2016-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        },
        'Italie': {
            'climat_affaires_observe': False,  # Italie : jamais observée
            'pib_base': 2200,
            'inflation_base': 1.8,
            'chomage_base': 10.5,
            'depenses_base': 50.0,
            'start_date': '2019-01-01',  # Début encore plus tardif
            'end_date': '2024-07-01',
            'prod_ind_start': '2019-06-01',  # Historique plus court
            'depenses_frequency': 'annuelle',
            'annual_start_date': '2016-01-01'  # Historique de la balance commerciale antérieur au début mensuel
        }
    }

    # Initialisation de la liste des jeux de données pour l'ensemble des pays
    all_data = []

    # Parcours des pays
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        # Création de l'index de dates propre à ce pays (début/fin distincts)
        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        # Création du DataFrame pour ce pays
        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle (mensuelle)
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise

        # Données non disponibles avant une certaine date
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation (mensuelle)
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage (mensuel)
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques (% du PIB) : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle
        # Historique disponible dès `annual_start_date`, antérieur au début des
        # variables mensuelles de ce pays : l'index temporel de l'entité en
        # devient irrégulier.
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Climat des affaires : enquête de conjoncture mensuelle publiée pour la
        # France et l'Allemagne, jamais observée pour l'Italie. La colonne existe
        # dans le frame pour les trois entités ; c'est l'entité italienne qui ne
        # l'observe jamais (support du paramètre covariate_eligibility et limite de
        # la stratégie 'interpolate' : high_frequency_imputer2_architecture.md §2.3
        # et §4.5). Réutilisation de la graine déjà en place, pas de second générateur.
        df_country['climat_affaires'] = np.nan
        if params['climat_affaires_observe']:
            climat_noise = np.random.normal(0, 2.0, n_periods)
            df_country.loc[dates, 'climat_affaires'] = 100.0 + climat_noise

        # Simulation des délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    # Concaténation et création du MultiIndex
    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel

### 2.3 — Construction et vue d'ensemble

In [ ]:
# Construction des deux jeux de données
df_timeseries = create_timeseries_dataset()
df_panel = create_panel_dataset()

# Affichage
print("=" * 90)
print("SÉRIES TEMPORELLES")
print("=" * 90)
print(f"Période : {df_timeseries.index.min():%Y-%m} à {df_timeseries.index.max():%Y-%m} "
      f"| {len(df_timeseries)} lignes")
print("\nNombre d'observations par colonne :")
print(df_timeseries.notna().sum().to_string())

print("\n" + "=" * 90)
print("PANEL")
print("=" * 90)
print(f"Entités : {df_panel.index.get_level_values('country').unique().tolist()} "
      f"| {df_panel.shape[0]} lignes, {df_panel.shape[1]} colonnes")
print("\nPériode couverte par entité :")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country:10s} : {dates_country.min():%Y-%m} → {dates_country.max():%Y-%m}")

print("\nNombre d'observations par (entité, colonne) :")
display(df_panel.groupby(level='country').apply(lambda g: g.notna().sum()).T)

In [ ]:
# Fonction de visualisation de la disponibilité des données
def plot_availability(frame: pd.DataFrame, title: str, ax=None):
    """Plot a NaN-pattern heatmap of a date-indexed frame.

    Args:
        frame: Frame to visualize, indexed by date only.
        title: Title of the axes.
        ax: Existing axes to draw on. A new figure is created when None.

    Returns:
        The axes the heatmap was drawn on.
    """
    # Création de la figure si aucun axe n'est fourni
    if ax is None:
        _, ax = plt.subplots(figsize=(14, 0.5 * len(frame.columns) + 1.5))

    # Matrice de disponibilité : 1 si la cellule est observée
    availability = frame.notna().astype(int)
    ax.imshow(availability.T, aspect='auto', interpolation='nearest',
              cmap=ListedColormap(['#f5d0d0', '#6ab04c']))

    # Configuration des axes
    ax.set_yticks(range(len(frame.columns)))
    ax.set_yticklabels(frame.columns, fontsize=8)
    ticks = np.linspace(0, len(frame) - 1, min(12, len(frame)), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{frame.index[i]:%Y-%m}" for i in ticks], rotation=45,
                       ha='right', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(False)
    return ax


# Disponibilité sur la série temporelle, puis sur chaque entité du panel
entities = df_panel.index.get_level_values('country').unique().tolist()
fig, axes = plt.subplots(len(entities) + 1, 1, figsize=(14, 2.6 * (len(entities) + 1)))
plot_availability(df_timeseries, "Séries temporelles", ax=axes[0])
for ax, country in zip(axes[1:], entities):
    plot_availability(df_panel.loc[country], f"Panel — {country}", ax=ax)

# Légende commune
fig.legend(handles=[mpatches.Patch(facecolor='#6ab04c', label='observée'),
                    mpatches.Patch(facecolor='#f5d0d0', label='manquante (NaN)')],
           loc='lower center', ncol=2, fontsize=9)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

In [ ]:
# Fréquences détectées, par colonne puis par (entité, colonne)
print("Séries temporelles — fréquence détectée par colonne :")
for column, frequency in detect_dataset_frequency(df_timeseries).items():
    print(f"  {column:32s} : {frequency}")

print("\nPanel — fréquence détectée par (entité, colonne) :")
detected_panel = pd.Series(detect_dataset_frequency(df_panel))
display(detected_panel.unstack(level=0).fillna('— jamais observée —'))

Deux points se lisent directement sur ce tableau, et ils commandent tout ce qui suit :

- `depenses_publiques_pib` est **annuelle pour la France et l'Italie, trimestrielle pour
  l'Allemagne**. Une même colonne, deux fréquences : chaque raisonnement de l'imputeur est donc
  mené par couple `(entité, colonne)`.
- `climat_affaires` n'a **aucune fréquence détectable pour l'Italie** : l'entité ne l'observe
  jamais.

<a id="3"></a>
## 3 — Première prise en main

Deux paramètres suffisent à faire tourner l'imputeur : la **fréquence cible** et l'**estimateur**.

```python
HighFrequencyImputer(target_frequency='M', estimator=LinearRegression())
```

Tout le reste a une valeur par défaut. Le schéma ci-dessous situe les paramètres les uns par
rapport aux autres : ce notebook les parcourt dans cet ordre.

In [ ]:
# Schéma d'ensemble : ce que chaque famille de paramètres gouverne
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Fonction auxiliaire de tracé d'une boîte annotée
def box(x, y, w, h, title, lines, color):
    """Draw a labelled rounded box listing parameters."""
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                                         facecolor=color, edgecolor='#2c3e50', linewidth=1.4))
    ax.text(x + w / 2, y + h - 0.38, title, ha='center', va='top',
            fontsize=10.5, fontweight='bold')
    for i, line in enumerate(lines):
        ax.text(x + 0.22, y + h - 0.95 - 0.42 * i, line, ha='left', va='top',
                fontsize=8.6, family='monospace')

# Les données d'entrée et la sortie
box(0.2, 7.6, 2.4, 1.9, "ENTRÉE", ["frames à", "fréquences", "mixtes"], '#ecf0f1')
box(7.4, 7.6, 2.4, 1.9, "SORTIE", ["valeurs +", "provenance", "par cellule"], '#ecf0f1')

# Les deux axes, au centre
box(0.2, 4.3, 4.6, 2.9, "AXE 1 — les COLONNES du modèle",
    ["covariate_strategy", "covariate_fallback", "covariate_eligibility",
     "interpolation_method / _anchor"], '#d6eaf8')
box(5.2, 4.3, 4.6, 2.9, "AXE 2 — les LIGNES de la cible",
    ["impute_intermediate_frequencies", "impute_unobserved_entities"], '#fdebd0')

# Les paramètres périphériques
box(0.2, 0.4, 3.0, 3.3, "FENÊTRES",
    ["imputation_scope", "coverage_threshold", "training_scope",
     "training_coverage_threshold"], '#e8f8f5')
box(3.6, 0.4, 3.0, 3.3, "ÉCHELLE & CONTRAINTES",
    ["scale_features", "aggregation_constraint", "additive_transformer"], '#e8f8f5')
box(7.0, 0.4, 2.8, 3.3, "ORDRE & SORTIE",
    ["fit_predict_order", "cv / cv_scoring", "keep_lower_frequencies",
     "restore_original_values"], '#e8f8f5')

# Flèches d'orientation
ax.annotate("", xy=(5.0, 7.2), xytext=(1.4, 7.55),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#2c3e50'))
ax.annotate("", xy=(8.6, 7.55), xytext=(7.4, 7.2),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#2c3e50'))
ax.set_title("Les paramètres du HighFrequencyImputer, par ce qu'ils gouvernent",
             fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

### 3.1 — Un premier ajustement

In [ ]:
# Ajustement minimal sur les séries temporelles
imputer_ts = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression())
result_ts = imputer_ts.fit_transform(df_timeseries)

# Affichage de l'état ajusté
print("Fréquence cible effective :", imputer_ts.effective_target_frequency_)
print("Progression de fréquences :", imputer_ts.frequency_progression_)
print("Fenêtre d'imputation      :", imputer_ts.imputation_window_)
print("Fenêtre d'entraînement    :", imputer_ts.training_window_)
print("\nCatégories de variables :")
for category, keys in imputer_ts.variable_categories_.items():
    print(f"  {category:12s} : {list(keys)}")
print(f"\nPlan figé : {len(imputer_ts.imputation_plan_)} étape(s)")
for step in imputer_ts.imputation_plan_:
    print(f"  étape {step.pred_freq_label} | {step.var_name:32s} "
          f"| source={step.source_frequency} | repli={step.is_fallback} "
          f"| provenance émise={step.emitted_provenance.value}")

Cet état ajusté expose les attributs :

- **`variable_categories_`** qui range chaque variable : `target_freq` (déjà à la fréquence cible, rien à faire), `impute` (à imputer), `aggregate` (plus fine que la cible, à agréger).
- **`frequency_progression_`** ne contient ici que `['M']` : sous le défaut `impute_intermediate_frequencies=False`, on va **directement** à la fréquence cible. C'est
  exactement ce que change l'axe 2 (section 5).
- **`imputation_plan_`** est l'état gelé au `fit` : une étape par couple (fréquence, variable).
  `transform` le **rejoue** sans jamais réajuster.

In [ ]:
# Fonction de comparaison visuelle avant / après imputation
def plot_before_after(original: pd.DataFrame, imputed: pd.DataFrame, columns,
                      title: str, start=None, end=None):
    """Plot observed anchors against the imputed high-frequency series.

    Args:
        original: Input frame, date-indexed.
        imputed: Imputed frame at the target frequency, date-indexed.
        columns: Columns to draw, one subplot each.
        title: Figure title.
        start: Left bound of the drawn period. None keeps everything.
        end: Right bound of the drawn period. None keeps everything.

    Returns:
        None. The figure is shown.
    """
    fig, axes = plt.subplots(len(columns), 1, figsize=(14, 3.1 * len(columns)), sharex=True)
    axes = np.atleast_1d(axes)

    # Tracé colonne par colonne
    for ax, column in zip(axes, columns):
        series_imputed = imputed[column].loc[start:end]
        series_original = original[column].loc[start:end].dropna()

        # Série imputée : la grille complète
        ax.plot(series_imputed.index, series_imputed.to_numpy(), color='#2980b9',
                linewidth=1.4, label='valeurs à la fréquence cible')
        # Observations d'origine : les ancres
        ax.scatter(series_original.index, series_original.to_numpy(), color='#c0392b',
                   zorder=5, s=38, label='observations d\'origine (ancres)')
        ax.set_ylabel(column, fontsize=9)
        ax.legend(fontsize=8, loc='upper left')

    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()


# Les deux variables imputées de la série temporelle
monthly_ts = result_ts.xs('M', level='frequency')
plot_before_after(df_timeseries, monthly_ts,
                  ['pib_trimestriel', 'balance_commerciale_annuelle'],
                  "Trimestriel et annuel portés au mensuel",
                  start='2019-01-01', end='2023-12-01')

### 3.2 — L'invariant additif

Sous le défaut `aggregation_constraint='sum'`, les sous-périodes imputées sont **recalées sur le
total observé** : la somme des mois d'un trimestre redonne exactement la valeur trimestrielle
publiée.

In [ ]:
# Vérification additive : somme des mois imputés contre la valeur trimestrielle observée
anchors = df_timeseries['pib_trimestriel'].dropna()
anchors = anchors[(anchors.index >= imputer_ts.imputation_window_[0])
                  & (anchors.index <= imputer_ts.imputation_window_[1])]

rows = []
for anchor_date, observed in anchors.items():
    # Les trois mois du trimestre ouvert par l'ancre
    period = monthly_ts['pib_trimestriel'].loc[
        anchor_date:anchor_date + pd.offsets.MonthBegin(2)]
    rows.append({
        'trimestre': f"{anchor_date:%Y-%m}",
        'valeur observée': round(observed, 3),
        'somme des mois imputés': round(float(period.sum()), 3),
        'écart': round(float(period.sum()) - observed, 10),
        'nb de mois': len(period),
    })

check = pd.DataFrame(rows)
display(check.head(8))
print("Écart maximal sur toute la fenêtre :", check['écart'].abs().max())

### 3.3 — Lire la sortie : `keep_lower_frequencies` et la provenance

`keep_lower_frequencies` est un **paramètre d'affichage pur** : il gouverne l'empilement des
niveaux de fréquence dans la sortie, jamais la logique. Les valeurs du niveau cible sont les mêmes
dans les deux cas.

In [ ]:
# Structure de sortie selon keep_lower_frequencies
for keep in (True, False):
    imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                                   keep_lower_frequencies=keep)
    out = imputer.fit_transform(df_timeseries)
    levels = (out.index.get_level_values('frequency').unique().tolist()
              if 'frequency' in (out.index.names or []) else ['— pas de niveau fréquence —'])
    print(f"keep_lower_frequencies={keep!s:5s} | index={out.index.names} "
          f"| niveaux={levels} | shape={out.shape}")

print("\nSous le défaut impute_intermediate_frequencies=False, la progression ne comporte que la")
print("fréquence cible : il n'y a donc aucun niveau intermédiaire à empiler (section 5).")

In [ ]:
# Fonction d'affichage de la matrice de provenance
PROVENANCE_ORDER = [
    ProvenanceType.ORIGINAL, ProvenanceType.AGGREGATED, ProvenanceType.INTERPOLATED,
    ProvenanceType.MODEL_ON_TRUE, ProvenanceType.MODEL_ON_INTERPOLATED,
    ProvenanceType.MODEL_ON_IMPUTED, ProvenanceType.MODEL_ON_IMPUTED_TARGET,
    ProvenanceType.MODEL_ON_IMPUTED_BOTH, ProvenanceType.MODEL_UNANCHORED,
]
PROVENANCE_COLORS = ['#bdc3c7', '#7f8c8d', '#f39c12', '#27ae60', '#f1c40f',
                     '#e67e22', '#e74c3c', '#8e44ad', '#2c3e50']


def provenance_labels(frame: pd.DataFrame) -> pd.DataFrame:
    """Turn a provenance matrix into a frame of readable strings."""
    return frame.map(lambda value: getattr(value, 'value', 'non imputé')
                     if value is not None and value == value else 'non imputé')


def target_level(matrix: pd.DataFrame, frequency: str = 'M') -> pd.DataFrame:
    """Restrict a matrix to one frequency level, or return it as is when flat.

    Args:
        matrix: Provenance matrix or output frame, stacked or not.
        frequency: Frequency level to read when the matrix is stacked.

    Returns:
        The frame restricted to that level, unchanged when it carries no
        frequency level (single-stage progression).
    """
    # Niveau de fréquence absent : la progression n'a qu'une étape
    if 'frequency' not in (matrix.index.names or []):
        return matrix
    return matrix.xs(frequency, level='frequency')


def plot_provenance(imputer, title: str, frequency: str = 'M', entity=None):
    """Plot the provenance matrix of a fitted imputer as a coloured heatmap.

    Args:
        imputer: Fitted HighFrequencyImputer.
        title: Title of the figure.
        frequency: Frequency level to read when the matrix is stacked.
        entity: Entity to restrict to on a panel. None keeps a time series as is.

    Returns:
        None. The figure is shown.
    """
    # Restriction au niveau de fréquence puis à l'entité demandée
    matrix = imputer.imputation_provenance_
    if 'frequency' in (matrix.index.names or []):
        matrix = matrix.xs(frequency, level='frequency')
    if entity is not None:
        matrix = matrix.xs(entity, level=0)

    # Encodage numérique, dans l'ordre de gravité croissante
    codes = {provenance.value: i for i, provenance in enumerate(PROVENANCE_ORDER)}
    labels = provenance_labels(matrix)
    numeric = labels.map(lambda name: codes.get(name, -1)).astype(float)

    # Tracé
    fig, ax = plt.subplots(figsize=(14, 0.45 * len(matrix.columns) + 2))
    cmap = ListedColormap(['#ffffff'] + PROVENANCE_COLORS)
    norm = BoundaryNorm(np.arange(-1.5, len(PROVENANCE_ORDER) + 0.5), cmap.N)
    ax.imshow(numeric.T, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')

    # Configuration des axes
    ax.set_yticks(range(len(matrix.columns)))
    ax.set_yticklabels(matrix.columns, fontsize=8)
    ticks = np.linspace(0, len(matrix) - 1, min(12, len(matrix)), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{matrix.index[i]:%Y-%m}" for i in ticks], rotation=45,
                       ha='right', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(False)

    # Légende restreinte aux provenances effectivement présentes
    present = set(labels.to_numpy().ravel())
    handles = [mpatches.Patch(facecolor='#ffffff', edgecolor='#bdc3c7', label='non imputé')]
    handles += [mpatches.Patch(facecolor=color, label=provenance.value)
                for provenance, color in zip(PROVENANCE_ORDER, PROVENANCE_COLORS)
                if provenance.value in present]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
    plt.tight_layout()
    plt.show()

    # Répartition chiffrée
    print("Répartition des provenances :")
    display(labels.apply(lambda col: col.value_counts()).fillna(0).astype(int))


plot_provenance(imputer_ts, "Provenance — séries temporelles, configuration par défaut")

La matrice de provenance répond, cellule par cellule, à la question « d'où vient cette valeur ? ».
Les neuf niveaux se lisent en trois familles :

| Famille | Niveaux | Sens |
|---|---|---|
| Non modèle | `original`, `aggregated`, `interpolated` | valeur d'entrée, agrégation additive exacte, ou interpolation |
| Modèle, par **pire ingrédient vu** | `model_on_true` → `model_on_interpolated` → `model_on_imputed` → `model_on_imputed_target` → `model_on_imputed_both` | de la plus sûre à la plus dégradée |
| Sans ancre | `model_unanchored` | l'entité n'observe jamais la colonne (section 5.3) |

C'est l'instrument de lecture de tout le reste du notebook : **chaque paramètre se voit dans cette
matrice** autant que dans les valeurs.

<a id="4"></a>
## 4 — Axe 1 : la matérialisation des covariables

> **La question de l'axe 1** : le modèle qui impute une variable a besoin de covariables sur
> **toute** la grille de prédiction. Que faire des covariables qui, elles aussi, sont observées
> moins souvent que cette grille ?

`covariate_strategy` a trois modalités :

| Modalité | Ce que le modèle reçoit |
|---|---|
| `'tolerate_nan'` | les covariables **telles qu'observées**, trous compris — l'estimateur doit tolérer les NaN |
| `'interpolate'` (défaut) | les covariables **interpolées** entre leurs ancres |
| `'model'` | les covariables **imputées par modèle**, en cascade sur les variables dans l'ordre `fit_predict_order` |

In [ ]:
# Schéma de l'axe 1 : ce que voit le modèle selon la modalité
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
grid = np.arange(12)
anchors = np.array([0, 3, 6, 9])

# Modalité 'tolerate_nan' : les ancres et des trous
axes[0].scatter(anchors, [1] * len(anchors), color='#c0392b', s=70, zorder=5, label='observée')
axes[0].scatter([m for m in grid if m not in anchors], [1] * (len(grid) - len(anchors)),
                facecolors='none', edgecolors='#bdc3c7', s=70, label='NaN transmis')
axes[0].set_title("'tolerate_nan'\nles trous sont transmis", fontsize=10, fontweight='bold')

# Modalité 'interpolate' : la courbe entre les ancres
axes[1].plot(grid, np.interp(grid, anchors, [1, 1, 1, 1]), color='#f39c12', linewidth=2,
             label='interpolée')
axes[1].scatter(anchors, [1] * len(anchors), color='#c0392b', s=70, zorder=5, label='ancre')
axes[1].set_title("'interpolate' (défaut)\nles trous sont comblés par interpolation",
                  fontsize=10, fontweight='bold')

# Modalité 'model' : les valeurs issues d'une imputation antérieure
axes[2].scatter(grid, [1] * len(grid), color='#e67e22', s=70, zorder=4, label='imputée par modèle')
axes[2].scatter(anchors, [1] * len(anchors), color='#c0392b', s=70, zorder=5, label='ancre')
axes[2].set_title("'model'\nles trous sont comblés par un modèle", fontsize=10, fontweight='bold')

# Mise en forme commune
for ax in axes:
    ax.set_ylim(0.6, 1.4)
    ax.set_yticks([])
    ax.set_xticks(grid)
    ax.set_xticklabels([f"M{i + 1}" for i in grid], fontsize=7)
    ax.legend(fontsize=7.5, loc='upper center', ncol=2)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle("Axe 1 — une covariable trimestrielle vue sur une grille mensuelle",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.1 — Voir ce que le modèle reçoit réellement

Un **estimateur espion** rend visible ce qui se passe à l'intérieur : il retient le `X_train` et le
`y_train` de chaque ajustement, ainsi que les trames de prédiction. Il tolère les NaN, ce qui le
rend utilisable sous les trois modalités.

In [ ]:
# Estimateur espion : il retient ce que chaque appel lui a montré
class SpyEstimator(BaseEstimator, RegressorMixin):
    """NaN-tolerant estimator predicting a constant, recording what it was shown."""

    def __init__(self, constant: float = 1.0):
        self.constant = constant

    def fit(self, X, y):
        """Record the training set and learn the mean of the target."""
        self.fit_X_ = X.copy()
        self.fit_y_ = y.copy()
        self.predict_X_ = []
        values = np.asarray(y, dtype=float)
        finite = values[~np.isnan(values)]
        self.mean_ = float(finite.mean()) if finite.size else 0.0
        return self

    def predict(self, X):
        """Record the prediction frame and return the learnt mean."""
        if not hasattr(self, 'predict_X_'):
            self.predict_X_ = []
        self.predict_X_.append(X.copy())
        return np.full(len(X), self.mean_)


def fit_imputer(data: pd.DataFrame, estimator=None, **params):
    """Fit an imputer on a dataset, with the notebook's shared defaults.

    Args:
        data: Dataset to fit on.
        estimator: Estimator handed to the imputer. Defaults to a SpyEstimator.
        **params: Any parameter of HighFrequencyImputer, overriding the defaults.

    Returns:
        The fitted imputer.
    """
    # Réglages par défaut du notebook, surchargeables par l'appelant
    settings = {'target_frequency': 'M',
                'estimator': SpyEstimator() if estimator is None else estimator}
    settings.update(params)
    imputer = HighFrequencyImputer(**settings)
    imputer.fit(data)
    return imputer


def describe_steps(imputer, title: str):
    """Print one line per plan step: training set, covariates and provenance."""
    print(f"--- {title} ---")
    for step in imputer.imputation_plan_:
        if step.is_fallback:
            print(f"  étape {step.pred_freq_label} | {step.var_name:32s} | "
                  f"REPLI PAR INTERPOLATION (aucun modèle ajusté)")
            continue
        model = step.model
        print(f"  étape {step.pred_freq_label} | {step.var_name:32s} | "
              f"X_train={model.fit_X_.shape} y_train={len(model.fit_y_)} | "
              f"provenance émise={step.emitted_provenance.value}")
        print(f"      covariables retenues : {list(model.fit_X_.columns)}")
        print(f"      voies de matérialisation : {dict(step.materialization)}")
        print(f"      NaN par covariable dans X_train : "
              f"{model.fit_X_.isna().sum().to_dict()}")


# Les trois modalités de l'axe 1, sur la série temporelle
for strategy in ('tolerate_nan', 'interpolate', 'model'):
    describe_steps(fit_imputer(df_timeseries, covariate_strategy=strategy),
                   f"covariate_strategy={strategy!r}")
    print()

La lecture est directe :

- **`'tolerate_nan'`** : `X_train` porte des NaN — les covariables basse fréquence n'ont de valeur
  qu'à leurs ancres. La voie de matérialisation est `raw_anchors`.
- **`'interpolate'`** : plus aucun NaN, les covariables sont interpolées (voie `interpolate`).
- **`'model'`** : les covariables déjà imputées à cette étape sont relues depuis le miroir
  (voies `stage_model` / `carried_model`), ce qui **chaîne** les imputations entre elles. C'est la
  seule modalité où `fit_predict_order` a un sens, puisque l'ordre décide qui sert de covariable à
  qui.

### 4.2 — Le prérequis dur de `'tolerate_nan'`

Le paramètre `'tolerate_nan'` utilise les covariables **telles quelles**, trous compris, pour entraîner le modèle. Un `LinearRegression` utilisé seuil lève une erreur sur ces NaN, et l'échec envoie **tout le groupe au repli par interpolation** : la modalité
n'a alors aucun effet visible, sinon celui de dégrader silencieusement le résultat. Il faut envelopper
l'estimateur dans un `Pipeline` portant un `SimpleImputer`, ou choisir une autre stratégie.

In [ ]:
# Le même réglage, avec et sans tolérance aux NaN dans l'estimateur
estimators = {
    'LinearRegression nu': LinearRegression(),
    'Pipeline(SimpleImputer, LinearRegression)': Pipeline([
        ('fill', SimpleImputer(strategy='mean')),
        ('reg', LinearRegression()),
    ]),
}

for label, estimator in estimators.items():
    imputer = fit_imputer(df_timeseries, estimator=estimator, covariate_strategy='tolerate_nan')
    fallbacks = sum(step.is_fallback for step in imputer.imputation_plan_)
    families = set(provenance_labels(
        target_level(imputer.imputation_provenance_)).to_numpy().ravel())
    print(f"{label:45s} | étapes en repli : {fallbacks}/{len(imputer.imputation_plan_)}"
          f" | provenances : {sorted(families - {'non imputé'})}")

### 4.3 — `covariate_fallback` : la voie de secours de `'model'`

Quand la voie « modèle » d'une covariable échoue — typiquement parce que rien n'a encore été imputé
pour elle à cette étape —, `covariate_fallback` décide de ce qui est utilisé à la place :
`'interpolate'` (défaut) ou `'tolerate_nan'`. **Ce paramètre est sans effet en dehors du cas où
`covariate_strategy='model'`.**

In [ ]:
# Effet du repli sous 'model', et son inertie sous 'interpolate'
for strategy in ('model', 'interpolate'):
    outputs = {}
    for fallback in ('interpolate', 'tolerate_nan'):
        imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                              covariate_strategy=strategy, covariate_fallback=fallback,
                              keep_lower_frequencies=False)
        outputs[fallback] = imputer.transform(df_timeseries)
    gap = (outputs['interpolate'] - outputs['tolerate_nan']).abs().max().max()
    print(f"covariate_strategy={strategy!r:13s} | écart maximal entre les deux replis : {gap:.6g}")

### 4.4 — `covariate_eligibility` : le panel et la colonne absente d'une entité

Sur un panel, une colonne peut être observée par certaines entités et **jamais** par d'autres :
c'est le cas de `climat_affaires`, absente pour l'Italie. Faut-il la garder comme covariable ?

| Modalité | Règle | Conséquence sur notre panel |
|---|---|---|
| `'any_entity'` (défaut) | la colonne est retenue dès qu'**une** entité l'observe | `climat_affaires` est retenue ; ses lignes italiennes restent NaN |
| `'all_entities'` | la colonne n'est retenue que si **toutes** les entités l'observent | `climat_affaires` est écartée du jeu de covariables |

Le choix est structurant sur un point : sous `'any_entity'`, un estimateur qui ne tolère pas les NaN
échoue sur les lignes italiennes et **toute l'étape part en repli**.

In [ ]:
# Les colonnes jugées éligibles par chaque modalité, lues sur le composant lui-même
for eligibility in ('any_entity', 'all_entities'):
    materializer = CovariateMaterializer(covariate_eligibility=eligibility)
    kept = list(materializer.eligible_columns(list(df_panel.columns), df_panel))
    print(f"covariate_eligibility={eligibility!r:14s} | colonnes éligibles : {kept}")
    print(f"    'climat_affaires' retenue : {'climat_affaires' in kept}")

In [ ]:
# Conséquence sur l'ajustement, avec un estimateur qui ne tolère PAS les NaN
for eligibility in ('any_entity', 'all_entities'):
    imputer = fit_imputer(df_panel, estimator=LinearRegression(),
                          covariate_strategy='model', covariate_eligibility=eligibility)
    fallbacks = sum(step.is_fallback for step in imputer.imputation_plan_)
    families = sorted(set(provenance_labels(
        target_level(imputer.imputation_provenance_)).to_numpy().ravel()) - {'non imputé'})
    print(f"covariate_eligibility={eligibility!r:14s} | étapes en repli : "
          f"{fallbacks}/{len(imputer.imputation_plan_)} | provenances : {families}")

print("\nSous 'any_entity', 'climat_affaires' est retenue, ses lignes italiennes sont NaN,")
print("LinearRegression lève, et TOUTES les étapes partent en repli par interpolation :")
print("plus aucune valeur ne vient d'un modèle.")

**Les trois options possibles** face à une telle colonne :

1. `covariate_eligibility='all_entities'` — la plus simple, au prix de l'information que la colonne
   apportait aux deux entités qui l'observent ;
2. un estimateur tolérant les NaN (`Pipeline` avec `SimpleImputer`, `HistGradientBoostingRegressor`) ;
3. un imputeur par entité, si les entités ne sont pas comparables (voir la mutualisation, § 5.4).

### 4.5 — `interpolation_method` et `interpolation_anchor`

Ces deux paramètres règlent la **forme** de l'interpolation, jamais sa direction. Ils acceptent
une valeur globale ou un dictionnaire par colonne.

- `interpolation_method` : la méthode passée à pandas (`'linear'`, `'nearest'`, `'cubic'`, ...).
- `interpolation_anchor` : la position de la valeur **dans sa période**, dans `[0, 1]`. `None`
  conserve l'ancrage détecté.

In [ ]:
# Comparaison visuelle des méthodes et des ancrages, sur la balance commerciale annuelle
column = 'balance_commerciale_annuelle'
window = slice('2019-01-01', '2023-12-01')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Effet de la méthode d'interpolation
for method, color in zip(('linear', 'nearest', 'cubic'), ('#2980b9', '#27ae60', '#8e44ad')):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='interpolate', interpolation_method=method,
                          keep_lower_frequencies=False)
    series = imputer.transform(df_timeseries)[column].loc[window]
    axes[0].plot(series.index, series.to_numpy(), color=color, linewidth=1.5,
                 label=f"interpolation_method={method!r}")

# Effet de l'ancrage dans la période
for anchor, color in zip((0.0, 0.5, 1.0), ('#e67e22', '#16a085', '#c0392b')):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='interpolate', interpolation_anchor=anchor,
                          keep_lower_frequencies=False)
    series = imputer.transform(df_timeseries)[column].loc[window]
    axes[1].plot(series.index, series.to_numpy(), color=color, linewidth=1.5,
                 label=f"interpolation_anchor={anchor}")

# Ancres observées, communes aux deux panneaux
observed = df_timeseries[column].loc[window].dropna()
for ax in axes:
    ax.scatter(observed.index, observed.to_numpy() / 12, color='#2c3e50', s=45, zorder=5,
               label="ancre observée (÷ 12, échelle mensuelle)")
    ax.legend(fontsize=8, loc='upper left')
    ax.set_ylabel(column, fontsize=8)

axes[0].set_title("Forme de la reconstruction : méthode puis ancrage",
                  fontsize=12, fontweight='bold')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

> ⚠️ **Le regard vers l'aval de `'interpolate'`.** Une interpolation linéaire entre deux ancres lit
> l'ancre **future**. Une valeur produite pour mars 2021 à partir d'ancres annuelles de 2021 et 2022
> incorpore donc une information indisponible en temps réel. C'est **délibéré** — l'imputeur
> reconstruit l'histoire, il ne prévoit pas — mais cela interdit d'utiliser la sortie telle quelle
> pour simuler une exécution en temps réel. `interpolation_method` et `interpolation_anchor`
> règlent la forme de cette reconstruction, jamais sa direction. La section 8 en tire les
> conséquences pour la validation croisée.

<a id="5"></a>
## 5 — Axe 2 : les fréquences intermédiaires

> **La question de l'axe 2** : pour porter une variable **annuelle** sur une grille **mensuelle**,
> faut-il passer directement de l'annuel au mensuel, ou imputer d'abord à la fréquence trimestrielle intermédiaire ? Et le modèle
> final a-t-il le droit de s'entraîner sur les imputations que la variable a elle-même reçues aux
> étapes précédentes ?

`impute_intermediate_frequencies` a trois modalités. Elles se distinguent par **deux décisions
indépendantes** :

| Modalité | Progression de fréquences | Lignes admises dans `y_train` |
|---|---|---|
| `False` (défaut) | directe : `['M']` | les **ancres observées** seulement |
| `'covariates_only'` | en cascade : `['Q', 'M']` | les **ancres observées** seulement |
| `True` | en cascade : `['Q', 'M']` | ancres **et** imputations antérieures de la variable |

`False` et `'covariates_only'` appliquent le **même filtre** et ne diffèrent que par le plan ;
`'covariates_only'` et `True` construisent le **même plan** et ne diffèrent que par le filtre.

In [ ]:
# Schéma de l'axe 2 : la progression des fréquences
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Trajet direct, sous False
axes[0].annotate("", xy=(0.85, 0.5), xytext=(0.15, 0.5),
                 arrowprops=dict(arrowstyle='->', lw=2.5, color='#2980b9'))
axes[0].text(0.15, 0.58, "Annuel", ha='center', fontsize=11, fontweight='bold')
axes[0].text(0.85, 0.58, "Mensuel", ha='center', fontsize=11, fontweight='bold')
axes[0].text(0.5, 0.36, "un seul saut\ny_train = 5 ancres annuelles", ha='center', fontsize=9)
axes[0].set_title("impute_intermediate_frequencies=False", fontsize=11, fontweight='bold')

# Trajet en cascade, sous 'covariates_only' et True
for x0, x1 in ((0.15, 0.5), (0.5, 0.85)):
    axes[1].annotate("", xy=(x1 - 0.03, 0.5), xytext=(x0 + 0.03, 0.5),
                     arrowprops=dict(arrowstyle='->', lw=2.5, color='#e67e22'))
for x, label in ((0.15, "Annuel"), (0.5, "Trimestriel"), (0.85, "Mensuel")):
    axes[1].text(x, 0.58, label, ha='center', fontsize=11, fontweight='bold')
axes[1].text(0.5, 0.30, "deux étapes : les valeurs trimestrielles produites\n"
                        "alimentent l'étape mensuelle\n"
                        "y_train = 5 ancres  ('covariates_only')  ou  20 lignes  (True)",
             ha='center', fontsize=9)
axes[1].set_title("impute_intermediate_frequencies='covariates_only' / True",
                  fontsize=11, fontweight='bold')

for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0.2, 0.8)
    ax.axis('off')
fig.suptitle("Axe 2 — le trajet d'une variable annuelle vers la grille mensuelle",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 — Ce que chaque modalité change dans le plan et dans `y_train`

L'estimateur espion rend la différence directement lisible : le nombre de lignes de `y_train`
de la balance commerciale, à l'étape mensuelle, passe de **5** (les ancres annuelles) à **20**
(ses propres imputations trimestrielles).

In [ ]:
# Les trois modalités de l'axe 2, sous covariate_strategy='model'
for modality in (False, 'covariates_only', True):
    imputer = fit_imputer(df_timeseries, covariate_strategy='model',
                          impute_intermediate_frequencies=modality)
    print(f"=== impute_intermediate_frequencies={modality!r} ===")
    print(f"    frequency_progression_ : {imputer.frequency_progression_}")
    for step in imputer.imputation_plan_:
        rows = '—' if step.is_fallback else len(step.model.fit_y_)
        print(f"    étape {step.pred_freq_label} | {step.var_name:32s} "
              f"| source={str(step.source_frequency):4s} | lignes de y_train : {rows} "
              f"| provenance émise={step.emitted_provenance.value}")
    print()

La provenance bascule sous `True` : la balance commerciale porte `model_on_imputed_target`, qui
décrit le mode d'entrainement du modèle d'imputation : **la cible d'entraînement contenait des valeurs produites par
un modèle**. La matrice de provenance décrit les valeurs qui ont été imputées ainsi.

### 5.2 — Effet sur les **valeurs**, et un cas limite instructif

Avec un `LinearRegression` non pénalisé, les valeurs produites sont ici **identiques** sous les
trois modalités. Ce n'est pas un défaut : les imputations trimestrielles de l'étape précédente se
trouvent exactement sur l'hyperplan ajusté à cette étape, donc les ré-apprendre redonne le même
hyperplan. Les coefficients ci-dessous le montrent.

Dès que l'estimateur est pénalisé (`Ridge`) ou non linéaire, l'égalité disparaît : le poids des
20 lignes n'est plus celui des 5 ancres.

In [ ]:
# Les coefficients du modèle mensuel de la balance commerciale, sous False puis True
for modality in (False, True):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', impute_intermediate_frequencies=modality)
    for step in imputer.imputation_plan_:
        if step.var_name == 'balance_commerciale_annuelle' and step.pred_freq_label == 'M':
            print(f"modalité={modality!r:17s} | coefficients : {np.round(step.model.coef_, 6)} "
                  f"| constante : {step.model.intercept_:.6f}")

In [ ]:
# Écart des valeurs produites, selon que l'estimateur est pénalisé ou non
def outputs_by_modality(estimator):
    """Return the imputed frames of the three axis-2 modalities."""
    frames = {}
    for modality in (False, 'covariates_only', True):
        imputer = fit_imputer(df_timeseries, estimator=estimator, covariate_strategy='model',
                              impute_intermediate_frequencies=modality,
                              keep_lower_frequencies=False)
        frames[modality] = imputer.transform(df_timeseries)
    return frames


for label, estimator in (('LinearRegression', LinearRegression()),
                         ('Ridge(alpha=10)', Ridge(alpha=10.0))):
    frames = outputs_by_modality(estimator)
    reference = frames[False]
    print(f"--- {label} : écart maximal des valeurs par rapport à False ---")
    for modality in ('covariates_only', True):
        gaps = (frames[modality] - reference).abs().max()
        moved = {column: round(float(gap), 4)
                 for column, gap in gaps.items() if gap == gap and gap > 1e-9}
        print(f"    {modality!r:17s} : {moved if moved else 'aucune valeur ne bouge'}")
    print()

### 5.3 — `'covariates_only'` : à quelle condition l'étape intermédiaire sert-elle ?

Sous `'covariates_only'`, la balance commerciale reçoit une étape **trimestrielle** en plus de
l'étape mensuelle. Ses valeurs trimestrielles n'ont **qu'un seul débouché** : servir de
**covariable** aux modèles des étapes suivantes. Elles n'entrent jamais dans le `y_train` de la
balance elle-même — c'est précisément ce qui distingue `'covariates_only'` de `True` (§ 5.1).

Pour qu'elles atteignent effectivement un modèle, **deux conditions** doivent être réunies :

1. **`covariate_strategy='model'`.** C'est la seule modalité qui lit le miroir des imputations.
   Sous `'interpolate'` ou `'tolerate_nan'`, une covariable est toujours servie depuis ses propres
   observations : l'étape trimestrielle est calculée, et visible dans la sortie empilée (§ 5.4),
   mais aucun modèle ne la lit.
2. **Aucune valeur plus fraîche n'existe à l'étape courante.** Lorsqu'un modèle mensuel cherche
   la balance comme covariable, deux voies sont possibles, dans cet ordre de priorité :

   | Voie | Valeur servie | Priorité |
   |---|---|---|
   | `stage_model` | la balance déjà imputée **à l'étape mensuelle** | 1 |
   | `carried_model` | la balance imputée à l'étape **trimestrielle**, reportée | 2 |

   Si la balance a déjà été imputée au mensuel, sa version trimestrielle est **supplantée** :
   c'est la même information, déjà exprimée à l'échelle de la grille.

La cellule suivante vérifie la condition 1.

In [ ]:
# Vérification chiffrée de l'inertie annoncée
for strategy in ('interpolate', 'tolerate_nan', 'model'):
    frames = {}
    for modality in (False, 'covariates_only'):
        imputer = fit_imputer(df_timeseries,
                              estimator=Pipeline([('fill', SimpleImputer()),
                                                  ('reg', LinearRegression())]),
                              covariate_strategy=strategy,
                              impute_intermediate_frequencies=modality,
                              keep_lower_frequencies=False)
        frames[modality] = imputer.transform(df_timeseries)
    gap = (frames['covariates_only'] - frames[False]).abs().max().max()
    verdict = "INERTE (valeurs identiques)" if gap < 1e-9 else f"actif (écart {gap:.4g})"
    print(f"covariate_strategy={strategy!r:14s} | 'covariates_only' vs False : {verdict}")

Hors de `covariate_strategy='model'`, la modalité est inopérante **par construction**. Sous `'model'`
elle l'est aussi **sur ce jeu**, mais pour une autre raison : la condition 2. Avec l'ordre par
défaut `fit_predict_order='frequency'`, la balance (annuelle) est imputée **avant** le PIB
(trimestriel) à l'étape mensuelle ; le modèle du PIB la lit donc par la voie `stage_model`, et la
version trimestrielle ne sert à rien.

Il suffit d'inverser l'ordre pour que l'étape intermédiaire devienne utile. Sous
`fit_predict_order='cv'`, le PIB passe en premier : la balance n'a pas encore été imputée au
mensuel quand son modèle est ajusté. Le tableau suit la voie par laquelle le modèle mensuel du PIB
reçoit la balance, sous les deux ordres.

In [ ]:
# Voie de la balance commerciale dans le modèle mensuel du PIB, selon l'ordre et la modalité
rows = []
for order in ('frequency', 'cv'):
    outputs = {}
    for modality in (False, 'covariates_only'):
        imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                              covariate_strategy='model', fit_predict_order=order,
                              cv=3, min_cv_train_size=3,
                              impute_intermediate_frequencies=modality,
                              keep_lower_frequencies=False)
        outputs[modality] = imputer.transform(df_timeseries)
        step = next(step for step in imputer.imputation_plan_
                    if step.var_name == 'pib_trimestriel' and step.pred_freq_label == 'M')
        rows.append({
            'fit_predict_order': order,
            'impute_intermediate_frequencies': repr(modality),
            "ordre à l'étape M": ' → '.join(imputer.imputation_order_['M']),
            'voie de la balance': step.materialization['balance_commerciale_annuelle'],
            'provenance du PIB': step.emitted_provenance.value,
        })
    # Écart des valeurs finales entre les deux modalités, pour cet ordre
    gap = float((outputs['covariates_only'] - outputs[False]).abs().max().max())
    rows[-2]['écart des valeurs / False'] = 0.0
    rows[-1]['écart des valeurs / False'] = round(gap, 4)

display(pd.DataFrame(rows))

**Lecture du tableau.**

- **Ordre `'frequency'`** : la balance est servie en `stage_model` sous les deux modalités. La
  valeur trimestrielle est supplantée, et les valeurs finales sont identiques.
- **Ordre `'cv'`** : sous `False`, la balance n'existe au mensuel que par **interpolation** de ses
  ancres annuelles (voie `interpolate`). Sous `'covariates_only'`, le modèle du PIB lit sa version
  **imputée au trimestriel** (voie `carried_model`), et les valeurs finales bougent.

**Règle à retenir** : `'covariates_only'` est inopérante hors de `covariate_strategy='model'`. Sous
`'model'`, il n'agit que lorsqu'une covariable basse fréquence est imputée **après** la variable
qui l'utilise. Il remplace alors une interpolation par une imputation intermédiaire.

### 5.4 — La sortie multi-fréquences

Quand la progression comporte plusieurs étapes, `keep_lower_frequencies=True` empile chaque niveau
dans la sortie. Le niveau de fréquence se place **du côté entité** de l'index : `(frequency, date)`
sur une série, `(entity..., 'frequency', 'date')` sur un panel.

In [ ]:
# Sortie empilée sous une progression à deux étapes
imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                               covariate_strategy='model',
                               impute_intermediate_frequencies='covariates_only',
                               keep_lower_frequencies=True)
stacked = imputer.fit_transform(df_timeseries)

print("Index :", stacked.index.names)
print("Niveaux empilés :", stacked.index.get_level_values('frequency').unique().tolist())
print("Lignes par niveau :")
print(stacked.groupby(level='frequency').size().to_string())

print("\nNiveau trimestriel (extrait) :")
display(stacked.xs('Q', level='frequency').loc['2021-01-01':'2021-12-01'].round(2))
print("Niveau mensuel (extrait) :")
display(stacked.xs('M', level='frequency').loc['2021-01-01':'2021-06-01'].round(2))

### 5.5 — `impute_unobserved_entities` : imputer une entité qui n'observe jamais la colonne

Ce paramètre est le pendant, **côté cible**, de `covariate_eligibility`. Il décide si une entité pour laquelle une colonne n'est **jamais** observée peut être totalement imputrée à partir des
autres entités du panel.

Les cellules ainsi produites portent `model_unanchored`, qui répond à une autre question que les
cinq niveaux `model_*` : non pas « quel est le pire ingrédient vu ? » mais « cette cellule est-elle
rattachée à une observation de sa propre colonne, pour sa propre entité ? ». Elles ne sont **ni
recalées ni divisées** : ce sont des prédictions libres, non la désagrégation d'un total observé.

In [ ]:
# Panel dérivé : l'Italie n'observe jamais les dépenses publiques
df_panel_absent = df_panel.copy()
df_panel_absent.loc['Italie', 'depenses_publiques_pib'] = np.nan

# Estimateur tolérant les NaN : l'entité sans ancre n'a pas d'interpolation de secours
tolerant = Pipeline([('fill', SimpleImputer()), ('reg', LinearRegression())])

for flag in (False, True):
    imputer = fit_imputer(df_panel_absent, estimator=tolerant, covariate_strategy='model',
                          impute_unobserved_entities=flag, keep_lower_frequencies=False)
    imputed = imputer.transform(df_panel_absent)
    series = imputed.loc['Italie', 'depenses_publiques_pib']
    provenance = imputer.imputation_provenance_.loc['Italie', 'depenses_publiques_pib']
    counts = provenance.dropna().map(
        lambda value: getattr(value, 'value', str(value))).value_counts().to_dict()
    print(f"impute_unobserved_entities={flag!s:5s} | cellules renseignées pour l'Italie : "
          f"{int(series.notna().sum())}/{len(series)}")
    print(f"    unanchored_pairs_ : {imputer.unanchored_pairs_}")
    print(f"    provenance        : {counts}")
    print(f"    frequency_progression_ inchangée : {imputer.frequency_progression_}")

In [ ]:
# Visualisation : ce que reçoit l'Italie, comparée aux deux entités qui observent la colonne
imputer = fit_imputer(df_panel_absent, estimator=tolerant, covariate_strategy='model',
                      impute_unobserved_entities=True, keep_lower_frequencies=False)
imputed = imputer.transform(df_panel_absent)

fig, ax = plt.subplots(figsize=(14, 4.5))
for country, color in zip(('France', 'Allemagne', 'Italie'),
                          ('#2980b9', '#27ae60', '#c0392b')):
    series = imputed.loc[country, 'depenses_publiques_pib'].dropna()
    style = '--' if country == 'Italie' else '-'
    ax.plot(series.index, series.to_numpy(), style, color=color, linewidth=1.6,
            label=f"{country}" + (" (sans aucune ancre)" if country == 'Italie' else ""))
    observed = df_panel_absent.loc[country, 'depenses_publiques_pib'].dropna()
    ax.scatter(observed.index, observed.to_numpy(), color=color, s=30, zorder=5)

ax.set_title("impute_unobserved_entities=True — l'Italie reçoit une imputation sans ancre",
             fontsize=12, fontweight='bold')
ax.set_ylabel('depenses_publiques_pib')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 5.6 — La mutualisation inter-entités, et son biais

Sur un panel, le jeu d'entraînement d'une variable rassemble **toutes les entités qui l'observent**,
chacune à la fréquence à laquelle elle l'observe, ramenée à l'échelle de l'étape. Deux conséquences
à garder en tête :

1. **Biais assumé** : mutualiser suppose des niveaux comparables entre entités. Un pays dix fois
   plus grand tire la cible, et `scale_features` ne corrige que l'échelle **de fréquence**, jamais
   celle des entités. La porte de sortie ne demande aucun paramètre : ajuster un imputeur par entité.
2. **La provenance est contagieuse entre entités** : une entité qui apporte des cellules
   `interpolated` ou `model` dégrade la provenance de **toutes** les cellules produites par
   l'étape — y compris celles des autres entités.

In [ ]:
# Composition du jeu d'entraînement mutualisé, étape par étape
imputer = fit_imputer(df_panel, covariate_strategy='model',
                      covariate_eligibility='all_entities')
for step in imputer.imputation_plan_:
    if step.is_fallback:
        continue
    print(f"étape {step.pred_freq_label} | {step.var_name:30s} | entités de l'étape : "
          f"{[entity[0] for entity in (step.entities or ())]}")
    print(f"    blocs d'entraînement (entité → fréquence apportée) : "
          f"{ {entity[0]: freq for entity, freq in step.training_blocks.items()} }")
    print(f"    lignes mutualisées dans y_train : {len(step.model.fit_y_)}")

<a id="6"></a>
## 6 — Le croisement des deux axes

Les deux axes se composent sans se connaître : trois modalités chacun, neuf configurations. Le
tableau ci-dessous les parcourt toutes et résume, pour chacune, ce qui change réellement.

In [ ]:
# Matrice des neuf configurations
tolerant_estimator = Pipeline([('fill', SimpleImputer()), ('reg', Ridge(alpha=10.0))])
reference_output = None
rows = []

for strategy in ('tolerate_nan', 'interpolate', 'model'):
    for modality in (False, 'covariates_only', True):
        started = time.perf_counter()
        imputer = fit_imputer(df_timeseries, estimator=tolerant_estimator,
                              covariate_strategy=strategy,
                              impute_intermediate_frequencies=modality,
                              keep_lower_frequencies=False)
        output = imputer.transform(df_timeseries)
        if reference_output is None:
            reference_output = output

        # Familles de provenance effectivement émises
        families = sorted(set(provenance_labels(imputer.imputation_provenance_)
                              .to_numpy().ravel()) - {'non imputé', 'original'})
        rows.append({
            'covariate_strategy': strategy,
            'intermediate': repr(modality),
            'étapes du plan': len(imputer.imputation_plan_),
            'replis': sum(step.is_fallback for step in imputer.imputation_plan_),
            'progression': '→'.join(imputer.frequency_progression_),
            'écart / référence': round(float((output - reference_output).abs().max().max()), 4),
            'provenances': ', '.join(families),
            'durée (s)': round(time.perf_counter() - started, 2),
        })

matrix = pd.DataFrame(rows)
display(matrix)
print("Référence : la première ligne (tolerate_nan, False).")

In [ ]:
# Lecture graphique : les valeurs produites pour la balance commerciale, par configuration
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
window = slice('2020-01-01', '2022-12-01')

for ax, strategy in zip(axes, ('tolerate_nan', 'interpolate', 'model')):
    for modality, style, color in ((False, '-', '#2980b9'),
                                   ('covariates_only', '--', '#e67e22'),
                                   (True, ':', '#c0392b')):
        imputer = fit_imputer(df_timeseries, estimator=tolerant_estimator,
                              covariate_strategy=strategy,
                              impute_intermediate_frequencies=modality,
                              keep_lower_frequencies=False)
        series = imputer.transform(df_timeseries)['balance_commerciale_annuelle'].loc[window]
        ax.plot(series.index, series.to_numpy(), style, color=color, linewidth=1.6,
                label=f"intermediate={modality!r}")
    ax.set_title(f"covariate_strategy={strategy!r}", fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.legend(fontsize=7.5)

axes[0].set_ylabel('balance_commerciale_annuelle')
fig.suptitle("Les neuf configurations sur une même variable", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Ce que la matrice montre** :

- l'axe 1 change le **nombre de replis** et les **familles de provenance** : c'est lui qui décide de
  ce que le modèle peut voir ;
- l'axe 2 change le **nombre d'étapes** et la **progression** ; il ne déplace les valeurs que
  sous `True`, la modalité qui touche à `y_train` ;
- `'covariates_only'` ne déplace aucune valeur hors de `covariate_strategy='model'`, exactement
  comme annoncé en 5.3.

<a id="7"></a>
## 7 — Les autres paramètres

Les deux axes décident de ce que le modèle voit. Les paramètres de cette section décident **sur
quelles lignes** il travaille, **à quelle échelle**, et **sous quelle contrainte**.

### 7.1 — Les fenêtres : `imputation_scope` et `coverage_threshold`

Trois fenêtres coexistent, et chacune a son masque :

| Attribut | Rôle |
|---|---|
| `strict_window_mask_` | fenêtre **stricte** : la période où toutes les variables sont disponibles. **Diagnostic seulement** — rien dans l'ajustement ne la lit. |
| `imputation_window_mask_` | fenêtre de **prédiction** : les lignes que l'imputeur écrit. Gouvernée par `imputation_scope`. |
| `training_window_mask_` | fenêtre d'**entraînement** : les lignes que les modèles voient. Suit `imputation_scope` par défaut, ou `training_scope` s'il est fourni. |

`imputation_scope` étend la fenêtre stricte : `'strict'` (défaut), `'extended_backward'`,
`'extended_forward'`, `'extended_both'`. `coverage_threshold` est le taux de couverture minimal
qu'une période doit atteindre pour être admise dans une extension.

In [ ]:
# Les quatre portées d'imputation, et leur effet sur la fenêtre
rows = []
for scope in ('strict', 'extended_backward', 'extended_forward', 'extended_both'):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', imputation_scope=scope)
    start, end = imputer.imputation_window_
    rows.append({
        'imputation_scope': scope,
        'début': f"{start:%Y-%m}",
        'fin': f"{end:%Y-%m}",
        'lignes imputées': int(imputer.imputation_window_mask_.sum()),
        'lignes strictes': int(imputer.strict_window_mask_.sum()),
        'lignes d\'entraînement': int(imputer.training_window_mask_.sum()),
    })
display(pd.DataFrame(rows))

In [ ]:
# Visualisation des fenêtres sur l'axe du temps
fig, ax = plt.subplots(figsize=(14, 4))
scopes = ('strict', 'extended_backward', 'extended_forward', 'extended_both')

for row, scope in enumerate(scopes):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', imputation_scope=scope)
    mask = imputer.imputation_window_mask_
    dates = mask.index
    ax.fill_between(dates, row - 0.35, row + 0.35,
                    where=mask.to_numpy(), color='#2980b9', alpha=0.75, step='mid')
    ax.fill_between(dates, row - 0.35, row + 0.35,
                    where=~mask.to_numpy(), color='#ecf0f1', step='mid')

# Repère de la fenêtre stricte
strict = fit_imputer(df_timeseries, estimator=LinearRegression(),
                     covariate_strategy='model').strict_window_mask_
strict_dates = strict.index[strict.to_numpy()]
ax.axvline(strict_dates.min(), color='#c0392b', linestyle='--', linewidth=1.4,
           label='bornes de la fenêtre stricte')
ax.axvline(strict_dates.max(), color='#c0392b', linestyle='--', linewidth=1.4)

ax.set_yticks(range(len(scopes)))
ax.set_yticklabels(scopes)
ax.set_title("Les quatre portées d'imputation", fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Effet du seuil de couverture sur une extension vers l'aval
rows = []
for threshold in (0.0, 0.25, 0.5, 0.75, 1.0):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', imputation_scope='extended_forward',
                          coverage_threshold=threshold)
    start, end = imputer.imputation_window_
    rows.append({'coverage_threshold': threshold, 'début': f"{start:%Y-%m}",
                 'fin': f"{end:%Y-%m}",
                 'lignes imputées': int(imputer.imputation_window_mask_.sum())})
display(pd.DataFrame(rows))
print("Plus le seuil est élevé, plus l'extension est exigeante : la fenêtre se resserre.")
print("À 1.0, aucune extension n'est admise : on retombe sur la fenêtre stricte.")

#### Étendre la fenêtre jusqu'aux observations isolées : `extended_both` à seuil nul

Sur ce jeu, la balance commerciale remonte à **2015**, alors que les variables mensuelles ne
commencent qu'en **2018**. Sur 2015-2017, l'index d'entrée ne porte donc que **trois dates**, les
observations annuelles. Avec `coverage_threshold=0.0`, n'importe quelle couverture suffit : la
fenêtre remonte jusqu'à ces dates, et l'imputeur travaille sur la grille mensuelle **densifiée**,
qui crée les mois absents de l'entrée.

La démonstration compare quatre réglages :

| Réglage | Question |
|---|---|
| `'strict'`, `LinearRegression` | la référence : que devient 2015-2017 sans extension ? |
| `'extended_both'` à seuil nul, `LinearRegression` nu | l'extension suffit-elle ? |
| `'extended_both'` à seuil nul, estimateur tolérant les NaN | que produit un modèle sur ces années ? |
| idem, avec `training_scope='strict'` | peut-on prédire sur l'extension sans s'y entraîner ? |

In [ ]:
# extended_both à seuil nul : quatre réglages comparés sur les années 2015-2017
EARLY = slice('2015-01-01', '2017-12-01')
tolerant_linear = Pipeline([('fill', SimpleImputer()), ('reg', LinearRegression())])

configurations = {
    "strict | LinearRegression": dict(
        estimator=LinearRegression(), imputation_scope='strict'),
    "extended_both, seuil 0 | LinearRegression": dict(
        estimator=LinearRegression(), imputation_scope='extended_both', coverage_threshold=0.0),
    "extended_both, seuil 0 | tolérant": dict(
        estimator=tolerant_linear, imputation_scope='extended_both', coverage_threshold=0.0),
    "extended_both, seuil 0 | tolérant, training_scope='strict'": dict(
        estimator=tolerant_linear, imputation_scope='extended_both', coverage_threshold=0.0,
        training_scope='strict'),
}

extended_outputs = {}
rows = []
for label, params in configurations.items():
    imputer = fit_imputer(df_timeseries, covariate_strategy='model',
                          keep_lower_frequencies=False, **params)
    output = imputer.transform(df_timeseries)
    extended_outputs[label] = output
    early = provenance_labels(imputer.imputation_provenance_).loc[EARLY]

    # Contrôle additif sur les trois années isolées
    gaps = []
    for year in (2015, 2016, 2017):
        observed = df_timeseries.loc[f'{year}-01-01', 'balance_commerciale_annuelle']
        months = output.loc[f'{year}-01-01':f'{year}-12-01', 'balance_commerciale_annuelle']
        if months.notna().any():
            gaps.append(abs(float(months.sum()) - observed))

    start, end = imputer.imputation_window_
    rows.append({
        'réglage': label,
        "fenêtre d'imputation": f"{start:%Y-%m} → {end:%Y-%m}",
        "fenêtre d'entraînement": "{:%Y-%m} → {:%Y-%m}".format(*imputer.training_window_),
        'étapes en repli': f"{sum(step.is_fallback for step in imputer.imputation_plan_)}"
                           f"/{len(imputer.imputation_plan_)}",
        'balance 2015-2017': early['balance_commerciale_annuelle'].value_counts().to_dict(),
        'PIB 2015-2017': early['pib_trimestriel'].value_counts().to_dict(),
        'écart additif max (2015-2017)': round(max(gaps), 10) if gaps else '—',
    })

added = extended_outputs["extended_both, seuil 0 | tolérant"].index.difference(df_timeseries.index)
print(f"Lignes d'entrée : {len(df_timeseries)} | mois absents de l'entrée sur la grille densifiée : "
      f"{len(added)} ({added.min():%Y-%m} → {added.max():%Y-%m})")
display(pd.DataFrame(rows).set_index('réglage'))

In [ ]:
# Visualisation : la balance commerciale mensuelle sur 2015-2019, selon le réglage
fig, ax = plt.subplots(figsize=(14, 4.5))
window = slice('2015-01-01', '2019-12-01')
styles = {
    "strict | LinearRegression": ('#7f8c8d', '-'),
    "extended_both, seuil 0 | LinearRegression": ('#e67e22', '--'),
    "extended_both, seuil 0 | tolérant": ('#2980b9', '-'),
}
for label, (color, style) in styles.items():
    series = extended_outputs[label]['balance_commerciale_annuelle'].loc[window]
    ax.plot(series.index, series.to_numpy(), style, color=color, linewidth=1.6, label=label)

# Ancres annuelles ramenées à l'échelle mensuelle
observed = df_timeseries['balance_commerciale_annuelle'].loc[window].dropna()
ax.scatter(observed.index, observed.to_numpy() / 12, color='#2c3e50', s=45, zorder=5,
           label="ancre observée (÷ 12, échelle mensuelle)")
ax.axvline(pd.Timestamp('2018-01-01'), color='#c0392b', linestyle=':', linewidth=1.4,
           label='début des variables mensuelles')

ax.set_title("extended_both à seuil nul : la grille mensuelle est créée avant 2018",
             fontsize=12, fontweight='bold')
ax.set_ylabel('balance_commerciale_annuelle', fontsize=9)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

**Ce que montre la comparaison.**

- **`'strict'`** : 2015-2017 sort de la fenêtre. Les trois ancres restent `original` et les mois
  intermédiaires ne sont pas imputés.
- **`'extended_both'` avec un `LinearRegression` nu** : la fenêtre d'entraînement suit celle
  d'imputation. Elle contient donc des lignes sans aucune variable mensuelle, l'estimateur lève,
  et **toutes les étapes partent en repli par interpolation**, y compris sur 2019-2023. L'extension
  dégrade alors tout le résultat, et pas seulement les années ajoutées.
- **Avec un estimateur tolérant les NaN**, les modèles sont bien ajustés et prédisent sur la
  grille densifiée :
  - la balance y porte une provenance `model_*` et, sous `aggregation_constraint='sum'`, les douze
    mois de chaque année isolée redonnent **exactement** son total publié ;
  - le PIB, qui n'a **aucune ancre** avant 2018, est prédit sans recalage, à partir de covariables
    mensuelles qui n'existent pas et que le `SimpleImputer` remplace par leur moyenne. La provenance
    `model_on_imputed` le signale : ces valeurs sont à lire avec prudence.
- **`training_scope='strict'`** garde l'entraînement sur la période où toutes les variables
  existent, et ne fait qu'**étendre la prédiction**. C'est le montage le plus sûr quand l'extension
  porte sur des périodes pauvres en covariables.

**À retenir** : un seuil nul étend la fenêtre jusqu'à la moindre observation, même isolée. C'est
utile pour reconstruire un long historique basse fréquence, à condition d'utiliser un estimateur
tolérant les NaN et, de préférence, de borner l'entraînement avec `training_scope`.

`training_scope` élargit la fenêtre **d'entraînement** seulement. Il **ajoute des lignes, jamais des
colonnes** : la sélection des covariables reste gouvernée par leur disponibilité **au moment de la
prédiction**. `training_coverage_threshold` joue le même rôle que `coverage_threshold` pour cette
fenêtre, et reste inopérant tant que `training_scope` n'est pas fourni.

In [ ]:
# Élargir la fenêtre d'entraînement sans toucher à celle de prédiction
for training_scope in (None, 'extended_both', 'unrestricted'):
    imputer = fit_imputer(df_timeseries, covariate_strategy='model',
                          training_scope=training_scope)
    rows = [len(step.model.fit_y_) for step in imputer.imputation_plan_
            if not step.is_fallback]
    print(f"training_scope={training_scope!r:16s} | lignes d'entraînement (masque) : "
          f"{int(imputer.training_window_mask_.sum()):3d} "
          f"| lignes imputées : {int(imputer.imputation_window_mask_.sum()):3d} "
          f"| y_train par étape : {rows}")

### 7.2 — `scale_features` : l'échelle de fréquence

Une valeur annuelle vaut douze valeurs mensuelles. `scale_features` décide comment ce facteur est
appliqué aux covariables et à la cible :

- `False` : aucune mise à l'échelle ;
- `'constant'` (défaut) : un diviseur constant (4 pour annuel→trimestriel, 12 pour annuel→mensuel) ;
- `'calendar'` : un diviseur calendaire, qui tient compte du nombre réel de sous-périodes ;
- un dictionnaire par colonne, pour panacher.

In [ ]:
# Influence de scale_features sur les coefficients du modèle, sous 'tolerate_nan' :
# la covariable annuelle y est servie à SES ancres, donc à son échelle
tolerant_linear = Pipeline([('fill', SimpleImputer()), ('reg', LinearRegression())])
outputs = {}
for mode in (False, 'constant', 'calendar'):
    imputer = fit_imputer(df_timeseries, estimator=tolerant_linear,
                          covariate_strategy='tolerate_nan', scale_features=mode,
                          keep_lower_frequencies=False)
    outputs[mode] = imputer.transform(df_timeseries)
    step = next(step for step in imputer.imputation_plan_
                if step.var_name == 'pib_trimestriel' and not step.is_fallback)
    coefficients = step.model.named_steps['reg'].coef_
    print(f"scale_features={mode!r:11s} | coefficients du modèle du PIB : "
          f"{np.round(coefficients, 4)}")

print("\nLe dernier coefficient est celui de la covariable annuelle. La mise à l'échelle divise")
print("cette covariable par son facteur (4 pour annuel → trimestriel) : le coefficient ajusté est")
print("donc multiplié d'autant, et les valeurs produites s'en trouvent déplacées.")
for mode in ('constant', 'calendar'):
    gap = (outputs[mode] - outputs[False]).abs().max().max()
    print(f"    écart maximal des valeurs, {mode!r} contre False : {gap:.6g}")
print("\nSur ce jeu, 'constant' et 'calendar' coïncident : les périodes y sont toutes complètes,")
print("donc le décompte calendaire des sous-périodes redonne le facteur constant.")

### 7.3 — `aggregation_constraint` : le recalage sur le total observé

`'sum'` (défaut) recale les sous-périodes imputées pour qu'elles **somment au total observé** ;
`None` laisse les prédictions telles quelles. Le paramètre accepte un dictionnaire par colonne.

> ⚠️ Sous `impute_intermediate_frequencies=True`, ce paramètre **cesse d'être orthogonal** à
> l'axe 2 : sous `'sum'`, une imputation antérieure tombant sur la date d'une ancre est retirée de
> `y_train` (le recalage en ferait une combinaison linéaire exacte des autres lignes) ; sous `None`
> elle est conservée et l'index d'entraînement gagne un niveau `frequency`.

In [ ]:
# Recalage additif : avec et sans contrainte
anchors = df_timeseries['pib_trimestriel'].dropna()
rows = []
for constraint in ('sum', None):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', aggregation_constraint=constraint,
                          keep_lower_frequencies=False)
    imputed = imputer.transform(df_timeseries)['pib_trimestriel']
    window_anchors = anchors[(anchors.index >= imputer.imputation_window_[0])
                             & (anchors.index <= imputer.imputation_window_[1])]
    gaps = []
    for anchor_date, observed in window_anchors.items():
        period = imputed.loc[anchor_date:anchor_date + pd.offsets.MonthBegin(2)]
        gaps.append(abs(float(period.sum()) - observed))
    rows.append({'aggregation_constraint': repr(constraint),
                 'écart additif maximal': round(max(gaps), 6),
                 'écart additif moyen': round(float(np.mean(gaps)), 6)})
display(pd.DataFrame(rows))
print("Sous 'sum', la somme des sous-périodes redonne exactement le total publié.")

### 7.4 — `fit_predict_order` : dans quel ordre imputer les variables

Sous `covariate_strategy='model'`, les variables se servent mutuellement de covariables : l'ordre
décide donc qui profite de qui. Deux modalités :

- `'frequency'` (défaut) : de la plus basse fréquence à la plus haute ;
- `'cv'` : chaque variable est subit une **validation croisée** sur le jeu même sur lequel elle sera ajustée, et
  les mieux prédites passent en premier.

`cv`, `cv_scoring` et `min_cv_train_size` ne servent que ce classement, et **tous sont inopérants hors
de `covariate_strategy='model'`**.

In [ ]:
# Les deux ordres d'imputation, et les scores qui décident du second
for order in ('frequency', 'cv'):
    imputer = fit_imputer(df_timeseries, estimator=LinearRegression(),
                          covariate_strategy='model', fit_predict_order=order,
                          cv=3, min_cv_train_size=3)
    print(f"fit_predict_order={order!r:11s} | ordre retenu : {dict(imputer.imputation_order_)}")
    if imputer.imputation_cv_scores_:
        for stage, scores in imputer.imputation_cv_scores_.items():
            print(f"    scores de l'étape {stage} : "
                  f"{ {name: round(score, 4) for name, score in scores.items()} }")

### 7.5 — `restore_original_values` et `inverse_transform`

`inverse_transform` défait la dernière passe. Sous `restore_original_values=True`, toute cellule
non NaN à l'entrée retrouve **exactement** sa valeur d'origine.

In [ ]:
# Aller-retour transform / inverse_transform
imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                               covariate_strategy='model', restore_original_values=True,
                               keep_lower_frequencies=False)
imputed = imputer.fit_transform(df_timeseries)
restored = imputer.inverse_transform(imputed)

# Comparaison aux valeurs d'entrée sur les cellules initialement observées
common = df_timeseries.index.intersection(restored.index)
original_cells = df_timeseries.loc[common].notna()
gap = (restored.loc[common][original_cells] - df_timeseries.loc[common][original_cells]).abs()
print(f"Cellules observées à l'entrée : {int(original_cells.to_numpy().sum())}")
print(f"Écart maximal après aller-retour : {float(gap.max().max()):.10g}")

### 7.6 — `verbose` et le suivi d'expériences

`verbose=True` journalise chaque phase de l'ajustement. Pour le suivi d'expériences,
`tsforecast.tracking.imputation_metrics` transforme l'état ajusté en un dictionnaire plat
directement loggable (`mlflow.log_metrics`).

In [ ]:
# Métriques plates, prêtes pour un tracker
from tsforecast.tracking import imputation_metrics

imputer = fit_imputer(df_timeseries, estimator=LinearRegression(), covariate_strategy='model')
metrics = imputation_metrics(imputer)
print(f"{len(metrics)} métriques exposées. Extrait :")
for name, value in list(metrics.items())[:12]:
    print(f"  {name:52s} : {value}")

<a id="8"></a>
## 8 — Intégration dans un workflow de prédiction

Cette section place l'imputeur dans une chaîne réaliste :

1. imputation des fréquences mixtes ;
2. application des **délais de publication** ;
3. **validation croisée**, et vérification qu'aucune information du futur ne fuit.

### 8.1 — Les délais de publication après l'imputation

L'imputeur reconstruit l'histoire ; le `PublicationDelayTransformer` rétablit ensuite ce qui était
**réellement connu** à une date de prédiction donnée. L'ordre compte : on impute d'abord sur
l'information complète, on applique ensuite le décalage de publication.

In [ ]:
# Imputation, puis application des délais de publication
imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                               covariate_strategy='model', keep_lower_frequencies=False)
imputed = imputer.fit_transform(df_timeseries)

# Délais typiques, en jours, référencés à la fin de la période
delays = pd.DataFrame({
    'column': ['pib_trimestriel', 'inflation_ipc', 'taux_chomage',
               'balance_commerciale_annuelle', 'production_industrielle'],
    'delay': [60.0, 30.0, 30.0, 90.0, 45.0],
    'unit': ['D'] * 5,
    'reference_point': ['end'] * 5,
})

delayer = PublicationDelayTransformer(delays=delays, strategy='shift')
delayed = delayer.fit_transform(imputed)

print("Dernière date renseignée par colonne :")
comparison = pd.DataFrame({
    'après imputation': imputed.apply(lambda col: col.last_valid_index()),
    'après délais': delayed.apply(lambda col: col.last_valid_index()),
})
display(comparison)

### 8.2 — Validation croisée : le bon montage

L'imputeur est un **transformateur sklearn conforme**. Deux points le rendent particulier dans une
chaîne prédictive :

1. il transforme **`X` et `y`** (il fusionne la cible pour l'imputer avec le reste) : il faut donc
   un `XYPipeline`, pas un `Pipeline` sklearn — ce dernier ne propage jamais `y` ;
2. comme il a vu `y` au `fit`, un appel à `predict(X)` seul lui réclame la colonne cible. Dans une
   validation croisée, on s'appuie donc sur la méthode `score` de la chaîne (le scoring par défaut
   de `cross_validate`), qui, elle, dispose de `y`.

Le point essentiel est ailleurs : **l'imputeur est réajusté à chaque pli**, donc il n'a jamais vu
le pli de test au moment où il apprend.

In [ ]:
# Préparation d'un problème de prédiction sur la fenêtre d'imputation
workflow = df_timeseries.loc['2019-01-01':'2023-12-01']
FEATURES = ['inflation_ipc', 'taux_chomage', 'pib_trimestriel',
            'balance_commerciale_annuelle']
TARGET = 'production_industrielle'

X_workflow = workflow[FEATURES]
y_workflow = workflow[TARGET]

# Découpage temporel : les plis se terminent sur des fins d'année, de sorte que la
# grille densifiée de chaque pli ne dépasse jamais ses propres observations
splitter = TSOutOfSampleSplit(n_splits=3, test_size=12, gap=0)
for fold, (train_idx, test_idx) in enumerate(splitter.split(X_workflow)):
    print(f"pli {fold} | train {X_workflow.index[train_idx][0]:%Y-%m} → "
          f"{X_workflow.index[train_idx][-1]:%Y-%m} ({len(train_idx)} mois)"
          f" | test {X_workflow.index[test_idx][0]:%Y-%m} → "
          f"{X_workflow.index[test_idx][-1]:%Y-%m} ({len(test_idx)} mois)")

In [ ]:
# La chaîne complète, l'imputeur réajusté à chaque pli
def build_pipeline():
    """Build the XY pipeline carrying the imputer, a NaN filler and a regressor."""
    return XYPipeline([
        ('imputer', HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                                         covariate_strategy='model',
                                         keep_lower_frequencies=False)),
        ('fill', SimpleImputer()),
        ('reg', Ridge(alpha=1.0)),
    ])


scores = cross_validate(build_pipeline(), X_workflow, y_workflow, cv=splitter,
                        error_score='raise')
print("R² par pli (scoring par défaut de la chaîne) :", np.round(scores['test_score'], 4).tolist())
print("Temps d'ajustement par pli (s) :", np.round(scores['fit_time'], 2).tolist())

### 8.3 — Vérification de l'absence de data leakage temporel

La vérification ne se fait pas sur le score : elle se fait sur les **valeurs imputées**. Le
protocole est direct — imputer deux fois la même fenêtre d'entraînement, une fois avec un imputeur
qui n'a vu que le passé, une fois avec un imputeur qui a vu tout l'échantillon. Si les valeurs
diffèrent, c'est que l'information du test a bien traversé.

In [ ]:
# Un imputeur ajusté sur TOUT l'échantillon : le montage fuyant
leaky_imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                                     covariate_strategy='model', keep_lower_frequencies=False)
leaky_output = leaky_imputer.fit_transform(workflow)

records = []
fold_series = {}

for fold, (train_idx, test_idx) in enumerate(splitter.split(X_workflow)):
    train_dates = X_workflow.index[train_idx]
    test_dates = X_workflow.index[test_idx]

    # Montage honnête : l'imputeur n'est ajusté que sur le passé du pli
    honest_imputer = HighFrequencyImputer(target_frequency='M', estimator=LinearRegression(),
                                          covariate_strategy='model',
                                          keep_lower_frequencies=False)
    honest_imputer.fit(workflow.loc[train_dates])
    # Le plan figé est rejoué sur tout l'échantillon : aucun réajustement
    honest_output = honest_imputer.transform(workflow)

    # Écart d'imputation sur la SEULE fenêtre d'entraînement
    gap = (honest_output.loc[train_dates, FEATURES]
           - leaky_output.loc[train_dates, FEATURES]).abs().max()

    # Scores des deux montages
    filler = SimpleImputer().fit(honest_output.loc[train_dates, FEATURES])
    honest_model = Ridge(alpha=1.0).fit(
        filler.transform(honest_output.loc[train_dates, FEATURES]), y_workflow.loc[train_dates])
    honest_mae = mean_absolute_error(
        y_workflow.loc[test_dates],
        honest_model.predict(filler.transform(honest_output.loc[test_dates, FEATURES])))

    leaky_filler = SimpleImputer().fit(leaky_output.loc[train_dates, FEATURES])
    leaky_model = Ridge(alpha=1.0).fit(
        leaky_filler.transform(leaky_output.loc[train_dates, FEATURES]),
        y_workflow.loc[train_dates])
    leaky_mae = mean_absolute_error(
        y_workflow.loc[test_dates],
        leaky_model.predict(leaky_filler.transform(leaky_output.loc[test_dates, FEATURES])))

    records.append({
        'pli': fold,
        'fin du train': f"{train_dates[-1]:%Y-%m}",
        'MAE — imputeur réajusté par pli': round(honest_mae, 3),
        'MAE — imputé une fois sur tout': round(leaky_mae, 3),
        'écart max des covariables imputées sur le train': round(float(gap.max()), 3),
    })
    fold_series[fold] = (train_dates, test_dates, honest_output, leaky_output)

display(pd.DataFrame(records))

In [ ]:
# Visualisation de la fuite : la même fenêtre d'entraînement, imputée deux fois
train_dates, test_dates, honest_output, leaky_output = fold_series[0]
column = 'pib_trimestriel'

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(honest_output.index, honest_output[column].to_numpy(), color='#2980b9', linewidth=1.6,
        label="imputeur ajusté sur le train du pli")
ax.plot(leaky_output.index, leaky_output[column].to_numpy(), color='#c0392b', linewidth=1.6,
        linestyle='--', label="imputeur ajusté sur tout l'échantillon")
observed = workflow[column].dropna()
ax.scatter(observed.index, observed.to_numpy(), color='#2c3e50', s=28, zorder=5,
           label='ancres observées')

# Zones d'entraînement et de test du pli
ax.axvspan(train_dates[0], train_dates[-1], color='#2ecc71', alpha=0.10)
ax.axvspan(test_dates[0], test_dates[-1], color='#e74c3c', alpha=0.10)
ax.text(train_dates[len(train_dates) // 2], ax.get_ylim()[1] * 0.98, 'ENTRAÎNEMENT',
        ha='center', va='top', fontsize=9, color='#27ae60', fontweight='bold')
ax.text(test_dates[len(test_dates) // 2], ax.get_ylim()[1] * 0.98, 'TEST',
        ha='center', va='top', fontsize=9, color='#c0392b', fontweight='bold')

ax.set_title("Les deux imputations diffèrent DANS la fenêtre d'entraînement :\n"
             "c'est la signature de l'information venue du test",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

**Ce que cette vérification établit — et ce qu'elle n'établit pas.**

L'écart mesuré sur la fenêtre d'entraînement n'est pas nul : ajuster l'imputeur sur tout
l'échantillon injecte bien de l'information postérieure dans des cellules antérieures. C'est la
définition même de la fuite, et elle est ici **chiffrée**, pas supposée.

Noter que la MAE du montage avec data leakage n'est pas nécessairement **meilleure**. C'est normal : une fuite ne garantit pas un meilleur score — elle rend le score **non
interprétable**, puisqu'il ne mesure plus ce que le modèle saurait faire en situation réelle.

**Trois règles pour une orchestration correcte :**

1. **L'imputeur se place à l'intérieur de la chaîne**, jamais avant la validation croisée. Ajusté
   par pli, il ne voit jamais le test au moment d'apprendre.
2. **`transform` ne réajuste rien** : le plan est figé au `fit`. Appliquer un imputeur ajusté sur le
   train à une fenêtre contenant le test est donc licite — c'est même le geste de production.
3. **Le regard vers l'aval de l'interpolation reste**, y compris dans un montage propre :
   une cellule interpolée entre deux ancres lit l'ancre future de sa propre fenêtre. L'imputeur
   reconstruit l'histoire ; il ne simule pas une exécution en temps réel. Pour cela, il faut lui
   adjoindre les délais de publication de la section 8.1.

<a id="9"></a>
## 9 — Récapitulatif

### 9.1 — Les paramètres, par ce qu'ils gouvernent

| Paramètre | Défaut | Ce qu'il change | Inopérant quand |
|---|---|---|---|
| `target_frequency` | — | la grille de sortie ; accepte un dict par entité | jamais |
| `estimator` | `None` | le modèle d'imputation ; `None` envoie tout à l'interpolation | jamais |
| `additive_transformer` | `None` | rend les données additives (log, différenciation) | jamais |
| **`covariate_strategy`** | `'interpolate'` | **axe 1** : les colonnes vues par le modèle | jamais |
| `covariate_fallback` | `'interpolate'` | la voie de secours de la route « modèle » | hors `covariate_strategy='model'` |
| `covariate_eligibility` | `'any_entity'` | l'agrégation de disponibilité sur les entités | hors panel |
| `interpolation_method` / `_anchor` | `'linear'` / `None` | la forme de la reconstruction | sous `'tolerate_nan'` |
| **`impute_intermediate_frequencies`** | `False` | **axe 2** : les lignes de `y_train` et la progression | `'covariates_only'` hors `'model'` |
| `impute_unobserved_entities` | `False` | l'imputation des couples (entité, colonne) jamais observés | hors panel |
| `fit_predict_order`, `cv`, `cv_scoring`, `min_cv_train_size` | `'frequency'`, ... | l'ordre d'imputation des variables | hors `covariate_strategy='model'` |
| `imputation_scope`, `coverage_threshold` | `'strict'`, `0.5` | la fenêtre de prédiction | jamais |
| `training_scope`, `training_coverage_threshold` | `None`, `None` | la fenêtre d'entraînement (lignes, jamais colonnes) | `training_coverage_threshold` sans `training_scope` |
| `scale_features` | `'constant'` | l'échelle de fréquence des features et de la cible | jamais |
| `aggregation_constraint` | `'sum'` | le recalage sur les totaux observés ; et `y_train` sous l'axe 2 `True` | jamais |
| `keep_lower_frequencies` | `True` | **l'affichage seul** : l'empilement des niveaux | progression à une seule étape |
| `restore_original_values` | `False` | le remplissage exact des cellules d'origine à l'inversion | jamais |
| `on_frequency_mismatch` | `'error'` | la réaction à une cible trop fine pour les données | jamais |
| `verbose` | `False` | la journalisation | jamais |

### 9.2 — Recommandations par cas d'usage

| Situation | Configuration conseillée |
|---|---|
| Première approche, jeu propre | les valeurs par défaut, `estimator=LinearRegression()` |
| Beaucoup de covariables basse fréquence | `covariate_strategy='model'`, `fit_predict_order='cv'` |
| Estimateur ne tolérant pas les NaN sur un panel troué | `covariate_eligibility='all_entities'`, ou `Pipeline(SimpleImputer, ...)` |
| Grand écart de fréquence (annuel → mensuel) | `impute_intermediate_frequencies='covariates_only'` **avec** `covariate_strategy='model'` |
| Historique court, peu d'ancres | `impute_intermediate_frequencies=True`, en acceptant `model_on_imputed_target` |
| Entité sans aucune observation d'une colonne | `impute_unobserved_entities=True` + estimateur tolérant les NaN |
| Sorties agrégées devant retomber sur les totaux publiés | `aggregation_constraint='sum'` (défaut) |
| Chaîne prédictive validée croisée | l'imputeur **dans** un `XYPipeline`, jamais en amont |

### 9.3 — Les cinq points à retenir

1. **Deux axes, deux questions.** L'axe 1 décide des **colonnes** vues par le modèle, l'axe 2 des
   **lignes** de sa cible. Ils se composent sans se connaître.
2. **La provenance est l'instrument de lecture.** Chaque paramètre s'y voit autant que dans les
   valeurs : une configuration qui produit `model_on_imputed_both` ne dit pas la même chose qu'une
   qui produit `model_on_true`.
3. **L'additivité est le contrat central de la classe.** Sous `aggregation_constraint='sum'`, la somme des sous-périodes
   redonne exactement le total publié ; `additive_transformer` est la seule porte de sortie.
4. **Un panel se raisonne par couple `(entité, colonne)`** : fréquences détectées, catégories,
   fenêtres. La mutualisation du jeu d'entraînement suppose des entités comparables, et propage la
   provenance de l'une à toutes.
5. **Le plan est figé au `fit`.** `transform` le rejoue sans jamais réapprendre : c'est ce qui rend
   l'objet utilisable en validation croisée — à condition de le placer **dans** la chaîne.

In [ ]:
# Temps total d'exécution du notebook
print(f"Notebook exécuté en {time.perf_counter() - NOTEBOOK_START:.1f} s.")